In [2]:
import sys
import os

# Add the src directory to sys.path
sys.path.append(os.path.abspath("."))

In [3]:
import os
import numpy as np
import scipy.io
import k3d
from anytree import Node, RenderTree

# -----------------------------
# 1. Set the model index
# -----------------------------
internal_idx = 1  # which .mat file to load
chair_dir = './dropbox/Chair/'

In [4]:
# -----------------------------
# 2. Load all necessary .mat files
# -----------------------------
ops_data = scipy.io.loadmat(os.path.join(chair_dir, 'ops', f'{internal_idx}.mat'))['op'][0]
labels_data = scipy.io.loadmat(os.path.join(chair_dir, 'labels', f'{internal_idx}.mat'))['label'][0]
boxes_data = scipy.io.loadmat(os.path.join(chair_dir, 'boxes', f'{internal_idx}.mat'))['box']
part_mesh_data = scipy.io.loadmat(os.path.join(chair_dir, 'part mesh indices', f'{internal_idx}.mat'))['cell_boxs_correspond_objSerialNumber'][0]
syms_data = scipy.io.loadmat(os.path.join(chair_dir, 'syms', f'{internal_idx}.mat'))['sym']
partnet_id = int(scipy.io.loadmat(os.path.join(chair_dir, 'syms', f'{internal_idx}.mat'))['shapename'][0])

print(f"PartNet ID: {partnet_id}")
print(f"Ops (node types): {ops_data}")
print(f"Labels (leaf nodes): {labels_data}")

PartNet ID: 1095
Ops (node types): [0 0 1 0 1]
Labels (leaf nodes): [1 2 0]


In [5]:
# -----------------------------
# 3. Identify leaf nodes
# -----------------------------
leaf_indices = [i for i, t in enumerate(ops_data) if t == 0]
leaf_nodes = []
for idx, leaf_idx in enumerate(leaf_indices):
    leaf_nodes.append({
        'idx': leaf_idx,
        'label': labels_data[idx],
        'box': boxes_data[leaf_idx],
        'mesh_indices': part_mesh_data[idx][0]
    })

In [6]:
# -----------------------------
# 4. Dynamically build tree
# -----------------------------
node_objects = [Node(f"Node{i} type={t} label={labels_data[leaf_indices.index(i)] if t==0 else None}") for i, t in enumerate(ops_data)]

# Build parent-child relationships dynamically
# We'll use a stack to build tree in depth-first order
stack = []
for i, t in enumerate(ops_data):
    if t == 0:  # leaf
        stack.append(node_objects[i])
    elif t == 1:  # adjacency
        right = stack.pop()
        left = stack.pop()
        node_objects[i].children = (left, right)
        stack.append(node_objects[i])
    elif t == 2:  # symmetry
        child = stack.pop()
        node_objects[i].children = (child,)
        stack.append(node_objects[i])
root_node = stack[0]

print("\nSymmetry tree:")
for pre, _, node in RenderTree(root_node):
    print(f"{pre}{node.name}")


Symmetry tree:
Node4 type=1 label=None
├── Node2 type=1 label=None
│   ├── Node0 type=0 label=1
│   └── Node1 type=0 label=2
└── Node3 type=0 label=0


In [7]:
# -----------------------------
# 5. Read OBB file
# -----------------------------
obb_file = os.path.join(chair_dir, 'obbs', f'{partnet_id}.obb')

def read_obb(file_path):
    obbs = []
    with open(file_path, 'r') as f:
        for line in f:
            line = line.strip()
            if line.startswith('N'):
                continue
            vals = line.split()
            if len(vals) < 12:
                continue
            obbs.append(np.array(list(map(float, vals[:12]))))
    return obbs

genshape = read_obb(obb_file)

In [8]:
# -----------------------------
# 6. Visualize OBBs in k3d
# -----------------------------
def make_box_k3d(obb, color=0xff0000, opacity=0.3):
    c = obb[0:3]
    l = obb[3:6]
    d1 = obb[6:9]; d1 /= np.linalg.norm(d1)
    d2 = obb[9:12]; d2 /= np.linalg.norm(d2)
    d3 = np.cross(d1, d2); d3 /= np.linalg.norm(d3)
    d1 *= l[0]/2
    d2 *= l[1]/2
    d3 *= l[2]/2

    corners = np.array([
        c - d1 - d2 - d3,
        c - d1 + d2 - d3,
        c + d1 - d2 - d3,
        c + d1 + d2 - d3,
        c - d1 - d2 + d3,
        c - d1 + d2 + d3,
        c + d1 - d2 + d3,
        c + d1 + d2 + d3,
    ], dtype=np.float32)

    indices = np.array([
        0,1,2, 0,2,3,
        4,5,6, 4,6,7,
        0,1,5, 0,5,4,
        2,3,7, 2,7,6,
        1,2,6, 1,6,5,
        0,3,7, 0,7,4
    ], dtype=np.uint32)

    return k3d.mesh(corners, indices, color=color, opacity=opacity)

plot = k3d.plot(grid_visible=True)
colors = [0xff0000, 0x00ff00, 0x0000ff, 0xffff00]

for i, leaf in enumerate(leaf_nodes):
    plot += make_box_k3d(genshape[i], color=colors[leaf['label'] % len(colors)])

plot.display()

Output()

In [9]:
obb_file = './dropbox/Chair/obbs/1095.obb'  # adjust if needed

with open(obb_file, 'r') as f:
    lines = f.readlines()

# Print the first 10 lines to inspect
for i, line in enumerate(lines[:10]):
    print(f"Line {i}: {line.strip()}")

Line 0: N 3
Line 1: 0.0302842 0.488309 -0.315397 -0.995024 0.0971271 -0.0222212 -0.00601635 0.164046 0.986434 0.0994548 0.981659 -0.162646 0.89214 0.305279 0.781198
Line 2: 0.00419755 -0.422335 0.169623 0.999916 0.0121707 -0.00438424 -0.0121707 0.770206 -0.637679 -0.00438424 0.637679 0.77029 0.898086 0.523685 1.15351
Line 3: 0.0030865 0.148677 0.0724135 1 0 0 0 1 0 0 0 1 0.890567 0.223845 0.805681
Line 4: C 2
Line 5: 0 2
Line 6: 1 2
Line 7: S 0
Line 8: L 3
Line 9: 0


In [10]:
import numpy as np

def read_obb_file(filepath):
    """
    Robust parser for PartNet .obb text files.
    
    - Skips header lines starting with 'N'
    - Skips metadata lines starting with 'C', 'S', 'L'
    - Reads only numeric lines
    - Takes first 12 floats per line: center(3), lengths(3), dir1(3), dir2(3)
    """
    obbs = []
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if len(line) == 0 or line.startswith(('N','C','S','L')):
                continue
            vals = line.split()
            if len(vals) < 12:
                # skip incomplete lines
                continue
            try:
                arr = np.array([float(v) for v in vals[:12]], dtype=np.float32)
                obbs.append(arr)
            except ValueError:
                # skip lines with non-numeric entries
                continue
    return obbs

# Example usage
obb_file = './dropbox/Chair/obbs/1095.obb'
genshape = read_obb_file(obb_file)
print(f"Read {len(genshape)} OBBs")
print("First OBB:", genshape[0])

Read 3 OBBs
First OBB: [ 0.0302842   0.488309   -0.315397   -0.995024    0.0971271  -0.0222212
 -0.00601635  0.164046    0.986434    0.0994548   0.981659   -0.162646  ]


In [11]:
from draw3dobb import showGenshape
showGenshape([genshape])  # wrap in a list if needed

AttributeError: 'list' object has no attribute 'squeeze'

<Figure size 640x480 with 0 Axes>

In [18]:
%matplotlib inline

import torch
genshape_torch = [torch.tensor(obb, dtype=torch.float32) for obb in genshape]
showGenshape(genshape_torch)

<Figure size 640x480 with 0 Axes>

In [16]:
def showGenshape_numpy(genshape):
    recover_boxes = genshape

    fig = plt.figure(0)
    cmap = plt.get_cmap('jet_r')
    ax = Axes3D(fig)
    ax.set_xlim(-0.7, 0.7)
    ax.set_ylim(-0.7, 0.7)
    ax.set_zlim(-0.7, 0.7)

    for jj in range(len(recover_boxes)):
        p = recover_boxes[jj]  # already a NumPy array
        draw(ax, p, cmap(float(jj)/len(recover_boxes)))

    plt.show()

# Use it
showGenshape_numpy(genshape)

NameError: name 'plt' is not defined

In [19]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # needed for 3D

def save_genshape_3d(genshape, filename='chair_obb.png'):
    fig = plt.figure(figsize=(8,8))
    ax = fig.add_subplot(111, projection='3d')
    ax.set_xlim(-0.7, 0.7)
    ax.set_ylim(-0.7, 0.7)
    ax.set_zlim(-0.7, 0.7)
    
    cmap = plt.get_cmap('jet_r')
    
    for jj in range(len(genshape)):
        p = genshape[jj]
        draw(ax, p, cmap(float(jj)/len(genshape)))
    
    # save the figure
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"3D OBB plot saved to {filename}")

# Example usage:
save_genshape_3d(genshape, 'chair_obb.png')

3D OBB plot saved to chair_obb.png
